# Part A — DNN Classification (Digits) using DataLoader (25 marks)

Q1. Digits DNN with mini-batch training (25)

Use sklearn.datasets.load_digits().

1. Load (X, y). Normalize X to [0, 1].
2. Create a train/val/test split:

* train = 70%
* val = 15%
* test = 15%
    (keep stratification and fixed random_state)

3. Create DataLoaders:

* train batch_size=64 shuffle=True
* val/test batch_size=256 shuffle=False

4. Build an MLP:

* 64 → 128 → 10 (one hidden layer only)

5. Train for 6 epochs. After each epoch print:

* train accuracy
* val accuracy

6. At the end print test accuracy.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
import numpy as np

# 1. Load and normalize
digits = load_digits()
X = digits.data.astype(np.float32) / 16.0   # pixel max is 16 → normalize to [0,1]
y = digits.target.astype(np.int64)

# 2. Train/Val/Test split  (70 / 15 / 15)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

print(f"Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}")

# 3. DataLoaders
train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
val_ds   = TensorDataset(torch.tensor(X_val),   torch.tensor(y_val))
test_ds  = TensorDataset(torch.tensor(X_test),  torch.tensor(y_test))

train_loader = DataLoader(train_ds, batch_size=64,  shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=256, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False)

# 4. Build MLP  64 → 128 → 10
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )
    def forward(self, x):
        return self.net(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MLP().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

def accuracy(loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = model(xb).argmax(dim=1)
            correct += (preds == yb).sum().item()
            total   += yb.size(0)
    return correct / total

# 5. Train for 6 epochs
for epoch in range(1, 7):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
    train_acc = accuracy(train_loader)
    val_acc   = accuracy(val_loader)
    print(f"Epoch {epoch}/6  |  Train Acc: {train_acc:.4f}  |  Val Acc: {val_acc:.4f}")

# 6. Test accuracy
test_acc = accuracy(test_loader)
print(f"\nTest Accuracy: {test_acc:.4f}")

# Part B — DataLoader + Augmentation (MNIST) (25 marks)

Q2. MNIST augmentation + visualize BEFORE/AFTER (25)

Use torchvision.datasets.MNIST.

1. Create train transforms that include:

* RandomAffine translate up to 10%
* ToTensor

2. Create test transforms with ToTensor only.
3. Load MNIST datasets and DataLoaders:

* train batch_size=128 shuffle=True
* test batch_size=256 shuffle=False

4. Pick one index (e.g., idx=0) and show:

* the original image (no augmentation)
* the augmented image (with augmentation)
    (2 subplots)

In [ ]:
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from PIL import Image

# 1. Train transforms: RandomAffine (translate 10%) + ToTensor
train_transform = transforms.Compose([
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor()
])

# 2. Test transforms: ToTensor only
test_transform = transforms.Compose([
    transforms.ToTensor()
])

# 3. Load MNIST
train_dataset = torchvision.datasets.MNIST(
    root='./data', train=True,  download=True, transform=train_transform
)
test_dataset  = torchvision.datasets.MNIST(
    root='./data', train=False, download=True, transform=test_transform
)

train_loader_mnist = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader_mnist  = DataLoader(test_dataset,  batch_size=256, shuffle=False)

print(f"Train size: {len(train_dataset)}  |  Test size: {len(test_dataset)}")

# 4. Visualize BEFORE / AFTER augmentation for idx=0
idx = 0

# Original (no augmentation) — load raw PIL image
original_dataset = torchvision.datasets.MNIST(
    root='./data', train=True, download=False, transform=transforms.ToTensor()
)
orig_img, label = original_dataset[idx]   # tensor C×H×W

# Augmented — apply train_transform
raw_pil_img = Image.fromarray(original_dataset.data[idx].numpy(), mode='L')
aug_img = train_transform(raw_pil_img)     # tensor C×H×W

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(orig_img.squeeze(), cmap='gray')
axes[0].set_title(f'Original (label={label})')
axes[0].axis('off')

axes[1].imshow(aug_img.squeeze(), cmap='gray')
axes[1].set_title('Augmented (RandomAffine)')
axes[1].axis('off')

plt.tight_layout()
plt.show()

# Part C — TensorBoard (15 marks)

Q3. TensorBoard: log scalars + one batch of images (15)

Using MNIST (from Part C or D):

1. Train for 2 epochs.
2. Log to TensorBoard:

* train loss per epoch
* test accuracy per epoch
* one batch of 16 images (once at epoch 1)

3. Write the TensorBoard command to run.


Logs must be under ./runs/

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
from torch.utils.tensorboard import SummaryWriter

# Reuse transforms from Part B
train_transform_tb = transforms.Compose([
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor()
])
test_transform_tb = transforms.Compose([transforms.ToTensor()])

train_ds_tb = torchvision.datasets.MNIST(
    root='./data', train=True,  download=True, transform=train_transform_tb
)
test_ds_tb  = torchvision.datasets.MNIST(
    root='./data', train=False, download=True, transform=test_transform_tb
)

train_loader_tb = DataLoader(train_ds_tb, batch_size=128, shuffle=True)
test_loader_tb  = DataLoader(test_ds_tb,  batch_size=256, shuffle=False)

# Simple CNN / MLP for MNIST  (1×28×28 → 784 → 128 → 10)
class MLP_MNIST(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )
    def forward(self, x):
        return self.net(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_tb = MLP_MNIST().to(device)
criterion_tb = nn.CrossEntropyLoss()
optimizer_tb = optim.Adam(model_tb.parameters(), lr=1e-3)

# TensorBoard writer — logs under ./runs/
writer = SummaryWriter(log_dir='./runs/mnist_tb')

def test_accuracy_tb(loader):
    model_tb.eval()
    correct = total = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = model_tb(xb).argmax(dim=1)
            correct += (preds == yb).sum().item()
            total   += yb.size(0)
    return correct / total

# Train for 2 epochs
for epoch in range(1, 3):
    model_tb.train()
    running_loss = 0.0
    for step, (xb, yb) in enumerate(train_loader_tb):
        xb, yb = xb.to(device), yb.to(device)
        optimizer_tb.zero_grad()
        out = model_tb(xb)
        loss = criterion_tb(out, yb)
        loss.backward()
        optimizer_tb.step()
        running_loss += loss.item()

        # Log one batch of 16 images at epoch 1, step 0
        if epoch == 1 and step == 0:
            img_grid = torchvision.utils.make_grid(xb[:16])
            writer.add_image('MNIST/first_batch_16', img_grid, global_step=0)

    avg_loss = running_loss / len(train_loader_tb)
    test_acc = test_accuracy_tb(test_loader_tb)

    # Log scalars
    writer.add_scalar('Loss/train', avg_loss, epoch)
    writer.add_scalar('Accuracy/test', test_acc, epoch)

    print(f"Epoch {epoch}/2  |  Train Loss: {avg_loss:.4f}  |  Test Acc: {test_acc:.4f}")

writer.close()
print("\nTensorBoard logs saved to ./runs/mnist_tb")

## TensorBoard Command

Run the following command in your terminal to launch TensorBoard:

```bash
tensorboard --logdir=./runs
```

Then open **http://localhost:6006** in your browser.